<a href="https://colab.research.google.com/github/changsin/AI/blob/main/06.3.astar_heuristic_search_gps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A* Search

## Graph class

In [ ]:
# A* Search
# 1. how to represent graph in a data structure - depends on the language and algorithm
# 2. how to test - depends very much on #1

# Finding routes
# each city is associated with a number - flying distance to Ulm
#   it also has a distance to its neighboring cities
# idea #1: what if we encode the distance to Ulm as a pair: e.g., (Ulm, 123) meaning

# how to implement a graph
# why graph? because that is where we can find relationship between nodes:
#   e.g., distance between cities

# Graph: a collection of connected nodes
#     a node: name - since the name is the only property of a node,
#       we can
#     an edge: distance
#   operations:
#
class Graph:
  def __init__(self, flight_connections, neighbors):
    self.flight_connections = flight_connections
    self.neighbors = neighbors

  def add_connection(self, conns, node1, node2, distance):
    if node1 not in conns:
      print("creating {} to {} {}".format(node2, node1, distance))
      conns.setdefault(node1, [(node2, distance)])
    else:
      print("adding {} to {} {}".format(node2, node1, distance))
      conns[node1].append((node2, distance))

    if node2 not in conns:
      print("creating {} to {} {}".format(node1, node2, distance))
      conns.setdefault(node2, [(node1, distance)])
    else:
      print("adding {} to {} {}".format(node1, node2, distance))
      conns[node2].append((node1, distance))

  def add_flight_connection(self, node1, node2, distance):
    self.add_connection(self.flight_connections, node1, node2, distance)

  def add_neighbor(self, node1, node2, distance):
    self.add_connection(self.neighbors, node1, node2, distance)

  def get_neighbors(self, node):
    if node in self.neighbors:
      return self.neighbors[node]

  def get_flight_distance(self, node1, node2):
    if node1 in self.flight_connections:
      for node, distance in self.flight_connections[node1]:
        if node == node2:
          return distance
    else:
      print("No connection between {} and {}".format(node1, node2))

graph = Graph(dict(), dict())

graph.add_flight_connection("Frankfurt", "Ulm", 215)
graph.add_flight_connection("Mannheim", "Ulm", 164)

print(graph.get_flight_distance("Ulm", "Frankfurt"))

print(graph.get_flight_distance("Frankfurt", "Ulm"))

print(graph.get_neighbors("Ulm"))

creating Ulm to Frankfurt 215
creating Frankfurt to Ulm 215
creating Ulm to Mannheim 164
adding Mannheim to Ulm 164
215
215
None


In [ ]:
graph = Graph(dict(), dict())

graph.add_flight_connection("Frankfurt", "Ulm", 215)
graph.add_flight_connection("Mannheim", "Ulm", 164)
graph.add_flight_connection("Karlsruhe", "Ulm", 137)
graph.add_flight_connection("Stuttgart", "Ulm", 75)
graph.add_flight_connection("Nurnberg", "Ulm", 132)
graph.add_flight_connection("Wurzburg", "Ulm", 153)

graph.add_neighbor("Mannheim", "Frankfurt", 85)
graph.add_neighbor("Mannheim", "Karlsruhe", 67)
graph.add_neighbor("Mannheim", "Nurnberg", 230)
graph.add_neighbor("Stuttgart", "Karlsruhe", 64)
graph.add_neighbor("Stuttgart", "Ulm", 107)
graph.add_neighbor("Frankfurt", "Wurzburg", 111)
graph.add_neighbor("Wurzburg", "Nurnberg", 104)
graph.add_neighbor("Wurzburg", "Ulm", 183)
graph.add_neighbor("Wurzburg", "Stuttgart", 140)
graph.add_neighbor("Nurnberg", "Ulm", 171)



print(graph.get_flight_distance("Ulm", "Frankfurt"))

print(graph.get_flight_distance("Karlsruhe", "Stuttgart"))
print(graph.get_neighbors("Ulm"))

creating Ulm to Frankfurt 215
creating Frankfurt to Ulm 215
creating Ulm to Mannheim 164
adding Mannheim to Ulm 164
creating Ulm to Karlsruhe 137
adding Karlsruhe to Ulm 137
creating Ulm to Stuttgart 75
adding Stuttgart to Ulm 75
creating Ulm to Nurnberg 132
adding Nurnberg to Ulm 132
creating Ulm to Wurzburg 153
adding Wurzburg to Ulm 153
creating Frankfurt to Mannheim 85
creating Mannheim to Frankfurt 85
adding Karlsruhe to Mannheim 67
creating Mannheim to Karlsruhe 67
adding Nurnberg to Mannheim 230
creating Mannheim to Nurnberg 230
creating Karlsruhe to Stuttgart 64
adding Stuttgart to Karlsruhe 64
adding Ulm to Stuttgart 107
creating Stuttgart to Ulm 107
adding Wurzburg to Frankfurt 111
creating Frankfurt to Wurzburg 111
adding Nurnberg to Wurzburg 104
adding Wurzburg to Nurnberg 104
adding Ulm to Wurzburg 183
adding Wurzburg to Ulm 183
adding Stuttgart to Wurzburg 140
adding Wurzburg to Stuttgart 140
adding Ulm to Nurnberg 171
adding Nurnberg to Ulm 171
215
None
[('Stuttgart', 10

In [ ]:
graph.flight_connections

{'Frankfurt': [('Ulm', 215)],
 'Karlsruhe': [('Ulm', 137)],
 'Mannheim': [('Ulm', 164)],
 'Nurnberg': [('Ulm', 132)],
 'Stuttgart': [('Ulm', 75)],
 'Ulm': [('Frankfurt', 215),
  ('Mannheim', 164),
  ('Karlsruhe', 137),
  ('Stuttgart', 75),
  ('Nurnberg', 132),
  ('Wurzburg', 153)],
 'Wurzburg': [('Ulm', 153)]}

In [ ]:
graph.neighbors

{'Frankfurt': [('Mannheim', 85), ('Wurzburg', 111)],
 'Karlsruhe': [('Mannheim', 67), ('Stuttgart', 64)],
 'Mannheim': [('Frankfurt', 85), ('Karlsruhe', 67), ('Nurnberg', 230)],
 'Nurnberg': [('Mannheim', 230), ('Wurzburg', 104), ('Ulm', 171)],
 'Stuttgart': [('Karlsruhe', 64), ('Ulm', 107), ('Wurzburg', 140)],
 'Ulm': [('Stuttgart', 107), ('Wurzburg', 183), ('Nurnberg', 171)],
 'Wurzburg': [('Frankfurt', 111),
  ('Nurnberg', 104),
  ('Ulm', 183),
  ('Stuttgart', 140)]}

## Heuristic Search
```
HEURISTICSEARCH(Start, Goal)
NodeList = [Start]
While True
  If NodeList = ∅
    Return(“No solution”)
  Node = First(NodeList)
  NodeList = Rest(NodeList)
  If GoalReached(Node, Goal)
    Return(“Solution found”, Node)
  NodeList = SortIn(Successors(Node),NodeList)
```

In [ ]:
def sort_in(to_sort):
  print("before sorted {}".format(to_sort))
  sorted_list = sorted(to_sort, key = lambda x: x[1], reverse = False)
  print("sorted {}".format(sorted_list))
  return sorted_list

sort_in(graph.get_neighbors("Ulm") + [])

before sorted [('Stuttgart', 107), ('Wurzburg', 183), ('Nurnberg', 171)]
sorted [('Stuttgart', 107), ('Nurnberg', 171), ('Wurzburg', 183)]


[('Stuttgart', 107), ('Nurnberg', 171), ('Wurzburg', 183)]

In [ ]:
def get_costs(to_sort, goal):
  print("before sorted {}".format(to_sort))
  sorted_list = sorted(to_sort, key = lambda x: x[1], reverse = False)
  print("sorted {}".format(sorted_list))
  return sorted_list

sort_in(graph.get_neighbors("Ulm") + [])

before sorted [('Stuttgart', 107), ('Wurzburg', 183), ('Nurnberg', 171)]
sorted [('Stuttgart', 107), ('Nurnberg', 171), ('Wurzburg', 183)]


[('Stuttgart', 107), ('Nurnberg', 171), ('Wurzburg', 183)]

In [ ]:
# find the shortest path between two cities using Greedy Best-First Search
def greedy_search(graph, city1, city2):
  if not city1 in graph.flight_connections or not city2 in graph.flight_connections:
    return None

  if city1 == city2:
    print("No need to find a route to a city itself")
    return [city1]

  # We retrieve neighbors as candidates
  neighbors = graph.get_neighbors(city1)
  if not neighbors:
    return None

  # Each item in to_search will be (node_name, flight_distance_to_goal)
  to_sort = []
  for name, _ in neighbors:
    h_val = graph.get_flight_distance(name, city2) or 0
    to_sort.append((name, h_val))

  to_search = sort_in(to_sort)
  paths = [city1]

  while to_search:
    print("to_search: {}".format(to_search))
    if to_search is None:
      print("no route")
      return

    # first check if the destination is in the immediate neighbor
    for node_name, _ in to_search:
      if node_name == city2:
        paths.append(node_name)
        print("Paths found {}".format(paths))
        return paths

    # if not found, add the first node with the lowest heuristic value to the paths
    node_name, _ = to_search[0]
    to_search = to_search[1:]
    paths.append(node_name)

    # Expand neighbors of the current node
    to_sort = []
    current_neighbors = graph.get_neighbors(node_name) or []
    # Combine with remaining search queue
    all_candidates = current_neighbors + [(node, 0) for node, _ in to_search]

    for key, _ in all_candidates:
      if key not in paths:
        h_val = graph.get_flight_distance(key, city2) or 0
        # Deduplicate candidates in list
        if not any(item[0] == key for item in to_sort):
          to_sort.append((key, h_val))
      else:
        print("visited {}".format(key))

    to_search = sort_in(to_sort)
    print("Found {} neighbors of {}".format(node_name, to_search))

In [ ]:
path = greedy_search(graph, "Mannheim", "Ulm")

if path:
    total_distance = 0
    for i in range(len(path) - 1):
        u, v = path[i], path[i+1]
        # Find edge cost between u and v
        neighbors = graph.get_neighbors(u) or []
        for neighbor, dist in neighbors:
            if neighbor == v:
                total_distance += dist
                break
    print(f"\nTotal actual distance traveled: {total_distance} km")

before sorted [('Frankfurt', 215), ('Karlsruhe', 137), ('Nurnberg', 132)]
sorted [('Nurnberg', 132), ('Karlsruhe', 137), ('Frankfurt', 215)]
to_search: [('Nurnberg', 132), ('Karlsruhe', 137), ('Frankfurt', 215)]
visited Mannheim
before sorted [('Wurzburg', 153), ('Ulm', 0), ('Karlsruhe', 137), ('Frankfurt', 215)]
sorted [('Ulm', 0), ('Karlsruhe', 137), ('Wurzburg', 153), ('Frankfurt', 215)]
Found Nurnberg neighbors of [('Ulm', 0), ('Karlsruhe', 137), ('Wurzburg', 153), ('Frankfurt', 215)]
to_search: [('Ulm', 0), ('Karlsruhe', 137), ('Wurzburg', 153), ('Frankfurt', 215)]
Paths found ['Mannheim', 'Nurnberg', 'Ulm']

Total actual distance traveled: 401 km


## A* Search

In [ ]:
def a_star_search(graph, start, goal):
    if start not in graph.neighbors or goal not in graph.flight_connections:
        print("Start or Goal node not in graph connections.")
        return None

    if start == goal:
        print("Start is already the goal!")
        return [start]

    # open_list will hold paths as tuples: (f_score, current_node, path_list, g_score)
    # We start with the start node. g(start) = 0
    h_start = graph.get_flight_distance(start, goal) if start != goal else 0
    if h_start is None:
        h_start = 0
    f_start = 0 + h_start

    open_list = [(f_start, start, [start], 0)]
    closed_set = set()

    print("Starting A* Search from {} to {}".format(start, goal))
    print("Initial Open List: {}".format(open_list))
    print("-" * 50)

    step = 1
    while open_list:
        # Sort open_list by f_score
        open_list.sort(key=lambda x: x[0])

        # Pop the node with the lowest f_score
        f_score, current, path, g_score = open_list.pop(0)

        print("\nStep {}:".format(step))
        print("  Popped node: '{}' with f(n) = {}".format(current, f_score))
        print("  Current Path: {}".format(path))
        print("  g(n) (cost from start): {}".format(g_score))

        if current == goal:
            print("\nGoal reached!")
            print("Final Path: {}".format(path))
            print("Total actual cost: {}".format(g_score))
            return path

        closed_set.add(current)

        # Get neighbors
        neighbors = graph.get_neighbors(current) or []
        print("  Neighbors of {}: {}".format(current, neighbors))

        for neighbor, distance_to_neighbor in neighbors:
            if neighbor in closed_set:
                continue

            # g(neighbor) = g(current) + step cost
            g_neighbor = g_score + distance_to_neighbor

            # h(neighbor) = flight distance to goal
            h_neighbor = graph.get_flight_distance(neighbor, goal)
            if h_neighbor is None:
                # Default to 0 if there is no heuristic available
                h_neighbor = 0

            # f(neighbor) = g(neighbor) + h(neighbor)
            f_neighbor = g_neighbor + h_neighbor

            print("    Evaluating neighbor: '{}'".format(neighbor))
            print("      g({}) = {} (path cost) + {} (edge) = {}".format(neighbor, g_score, distance_to_neighbor, g_neighbor))
            print("      h({}) = {} (flight distance to {})".format(neighbor, h_neighbor, goal))
            print("      f({}) = g({}) + h({}) = {}".format(neighbor, g_neighbor, h_neighbor, f_neighbor))

            # Check if this neighbor is already in open_list with a lower f_score
            existing_in_open = False
            for idx, (f_op, node_op, path_op, g_op) in enumerate(open_list):
                if node_op == neighbor:
                    existing_in_open = True
                    if g_neighbor < g_op:
                        print("      * Found shorter path to '{}', updating open list.".format(neighbor))
                        open_list[idx] = (f_neighbor, neighbor, path + [neighbor], g_neighbor)
                    break

            if not existing_in_open:
                open_list.append((f_neighbor, neighbor, path + [neighbor], g_neighbor))

        step += 1

    print("No route found.")
    return None

In [ ]:
# Test the A* Search algorithm implementation
path = a_star_search(graph, "Mannheim", "Ulm")

Starting A* Search from Mannheim to Ulm
Initial Open List: [(164, 'Mannheim', ['Mannheim'], 0)]
--------------------------------------------------

Step 1:
  Popped node: 'Mannheim' with f(n) = 164
  Current Path: ['Mannheim']
  g(n) (cost from start): 0
  Neighbors of Mannheim: [('Frankfurt', 85), ('Karlsruhe', 67), ('Nurnberg', 230)]
    Evaluating neighbor: 'Frankfurt'
      g(Frankfurt) = 0 (path cost) + 85 (edge) = 85
      h(Frankfurt) = 215 (flight distance to Ulm)
      f(Frankfurt) = g(85) + h(215) = 300
    Evaluating neighbor: 'Karlsruhe'
      g(Karlsruhe) = 0 (path cost) + 67 (edge) = 67
      h(Karlsruhe) = 137 (flight distance to Ulm)
      f(Karlsruhe) = g(67) + h(137) = 204
    Evaluating neighbor: 'Nurnberg'
      g(Nurnberg) = 0 (path cost) + 230 (edge) = 230
      h(Nurnberg) = 132 (flight distance to Ulm)
      f(Nurnberg) = g(230) + h(132) = 362

Step 2:
  Popped node: 'Karlsruhe' with f(n) = 204
  Current Path: ['Mannheim', 'Karlsruhe']
  g(n) (cost from start): 

In [ ]:
lists = [('Frankfurt', 85), ('Karlsruhe', 67), ('Wurzburg', 104), ('Ulm', 171)]

'Frankfurt' in [ name for name, value in lists]

True

# Week 3 Practice Quiz for Metrics.

In [ ]:
import numpy as np

A = np.array([[1, 2, 3],[4, 5, 6]])
B = np.array([1, 2, 3])

A * B

array([[ 1,  4,  9],
       [ 4, 10, 18]])

In [ ]:
A @ B

array([ 6, 15])

In [ ]:
A.shape

(2, 3)

In [ ]:
B.shape

(3,)

In [ ]:
np.multiply(A, B)

array([[1, 2, 3],
       [4, 5, 6]])

In [ ]:

class Node():

    def __init__(self, parent=None, position=None):
        self.parent = parent
        self.position = position

        self.g = 0
        self.h = 0
        self.f = 0

    def __eq__(self, other):
        return self.position == other.position


def astar(maze, start, end):
    #Returns a list of tuples as a path from the given start to the given end in the given maze

    # Create start and end node
    start_node = Node(None, start)
    start_node.g = start_node.h = start_node.f = 0
    end_node = Node(None, end)
    end_node.g = end_node.h = end_node.f = 0

    # Initialize both open and closed list
    open_list = []
    closed_list = []

    # Add the start node
    open_list.append(start_node)

    # Loop until you find the end
    while len(open_list) > 0:

        # Get the current node
        current_node = open_list[0]
        current_index = 0
        for index, item in enumerate(open_list):
            if item.f < current_node.f:
                current_node = item
                current_index = index

        # Pop current off open list, add to closed list
        open_list.pop(current_index)
        closed_list.append(current_node)

        # Found the goal
        if current_node == end_node:
            path = []
            current = current_node
            while current is not None:
                path.append(current.position)
                current = current.parent
            return path[::-1] # Return reversed path

        # Generate children
        children = []
        for new_position in [(0, -1), (0, 1), (-1, 0), (1, 0), (-1, -1), (-1, 1), (1, -1), (1, 1)]: # Adjacent squares

            # Get node position
            node_position = (current_node.position[0] + new_position[0], current_node.position[1] + new_position[1])

            # Make sure within range
            if node_position[0] > (len(maze) - 1) or node_position[0] < 0 or node_position[1] > (len(maze[len(maze)-1]) -1) or node_position[1] < 0:
                continue

            # Make sure walkable terrain
            if maze[node_position[0]][node_position[1]] != 0:
                continue

            # Create new node
            new_node = Node(current_node, node_position)

            # Append
            children.append(new_node)

        # Loop through children
        for child in children:

            # Child is on the closed list
            for closed_child in closed_list:
                if child == closed_child:
                    continue

            # Create the f, g, and h values
            child.g = current_node.g + 1
            child.h = ((child.position[0] - end_node.position[0]) ** 2) + ((child.position[1] - end_node.position[1]) ** 2)
            child.f = child.g + child.h

            # Child is already in the open list
            for open_node in open_list:
                if child == open_node and child.g > open_node.g:
                    continue

            # Add the child to the open list
            open_list.append(child)


def main():
    maze1 = [
                [0, 0, 1, 0, 0],
                [0, 1, 1, 0, 0],
                [0, 0, 0, 0, 0],
                [0, 0, 1, 0, 0],
                [0, 0, 1, 0, 0]
            ]
    maze2 = [
                [1, 0, 0, 0, 0],
                [1, 1, 0, 0, 0],
                [0, 1, 1, 0, 0],
                [0, 0, 0, 0, 0],
                [1, 1, 0, 0, 0]
            ]
    maze3 = [
                [0, 0, 0, 0, 0],
                [0, 1, 1, 0, 0],
                [0, 1, 1, 1, 0],
                [0, 1, 1, 0, 0],
                [0, 0, 0, 0, 0]
            ]

    maze = maze3

    start = (4,0)
    end = (1,3)

    path = astar(maze, start, end)
    print(path)


# if __name__ == '__main__':
#     main()


In [ ]:
main()

[(4, 0), (4, 1), (4, 2), (3, 3), (2, 4), (1, 3)]
